In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset

DATA_DIR = "processed_data"

def load_pickle(filename):
    with open(os.path.join(DATA_DIR, filename), "rb") as f:
        return pickle.load(f)

# --------------------------------------------------
# Load encoded datasets
# --------------------------------------------------

train_encodings = load_pickle("train_encodings.pkl")
val_encodings   = load_pickle("val_encodings.pkl")
test_encodings  = load_pickle("test_encodings.pkl")

# --------------------------------------------------
# Load chunk → original prompt mappings
# --------------------------------------------------

train_mapping = load_pickle("train_mapping.pkl")
val_mapping   = load_pickle("val_mapping.pkl")
test_mapping  = load_pickle("test_mapping.pkl")

print("Train chunks:", len(train_mapping))
print("Validation chunks:", len(val_mapping))
print("Test chunks:", len(test_mapping))

Train chunks: 334755
Validation chunks: 41833
Test chunks: 41744


In [2]:
class PromptChunkDataset(Dataset):

    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(
                self.encodings["input_ids"][idx],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                self.encodings["attention_mask"][idx],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                self.encodings["labels"][idx],
                dtype=torch.long
            )
        }


train_dataset = PromptChunkDataset(train_encodings)
val_dataset   = PromptChunkDataset(val_encodings)
test_dataset  = PromptChunkDataset(test_encodings)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 334755
Validation dataset: 41833
Test dataset: 41744


In [3]:
train_df = pd.read_csv("train.csv")
val_df   = pd.read_csv("validation.csv")
test_df  = pd.read_csv("test.csv")

y_val = val_df["label"].astype(int).to_numpy()
y_test = test_df["label"].astype(int).to_numpy()

print("Train prompts:", len(train_df))
print("Validation prompts:", len(val_df))
print("Test prompts:", len(test_df))

Train prompts: 290169
Validation prompts: 36271
Test prompts: 36272


In [5]:
from transformers import (
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed
)

MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"

SEEDS = [42, 123, 456]

LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
NUM_EPOCHS = 3
WEIGHT_DECAY = 0.01

In [6]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [7]:
models = []
trainers = []

for member_id, seed in enumerate(SEEDS, start=1):

    print("\n" + "=" * 80)
    print(f"TRAINING ENSEMBLE MEMBER {member_id}")
    print(f"SEED = {seed}")
    print("=" * 80)

    # --------------------------------------------------
    # Set seed BEFORE model initialization
    # --------------------------------------------------

    set_seed(seed)

    # --------------------------------------------------
    # Load pretrained BERT-Tiny
    # --------------------------------------------------

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    # --------------------------------------------------
    # Training arguments
    # --------------------------------------------------

    output_dir = f"./bert_tiny_ensemble_member_{member_id}"

    training_args = TrainingArguments(
        output_dir=output_dir,

        eval_strategy="epoch",
        save_strategy="epoch",

        learning_rate=LEARNING_RATE,

        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,

        num_train_epochs=NUM_EPOCHS,

        weight_decay=WEIGHT_DECAY,

        load_best_model_at_end=True,

        metric_for_best_model="f1",
        greater_is_better=True,

        logging_steps=500,

        save_total_limit=2,

        report_to="none",

        fp16=torch.cuda.is_available(),

        seed=seed
    )

    # --------------------------------------------------
    # Trainer
    # --------------------------------------------------

    trainer = Trainer(
        model=model,
        args=training_args,

        train_dataset=train_dataset,
        eval_dataset=val_dataset,

        compute_metrics=compute_metrics
    )

    # --------------------------------------------------
    # Train
    # --------------------------------------------------

    trainer.train()

    # --------------------------------------------------
    # Store
    # --------------------------------------------------

    models.append(model)
    trainers.append(trainer)

    print(f"\nFinished ensemble member {member_id}")


TRAINING ENSEMBLE MEMBER 1
SEED = 42


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Use

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.118903,0.108283,0.958956,0.970425,0.952468,0.961363
2,0.102340,0.096073,0.965577,0.973170,0.962322,0.967716
3,0.083985,0.094940,0.967227,0.975047,0.963526,0.969253


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Finished ensemble member 3


In [8]:
def softmax(x):

    x = x - np.max(
        x,
        axis=1,
        keepdims=True
    )

    exp_x = np.exp(x)

    return exp_x / np.sum(
        exp_x,
        axis=1,
        keepdims=True
    )


val_member_probs = []

for member_id, trainer in enumerate(trainers, start=1):

    print(
        f"Generating validation predictions "
        f"for member {member_id}..."
    )

    output = trainer.predict(val_dataset)

    logits = output.predictions

    probs = softmax(logits)[:, 1]

    val_member_probs.append(probs)

    print(
        f"Member {member_id} logits shape:",
        logits.shape
    )

Generating validation predictions for member 1...


C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Member 1 logits shape: (41833, 2)
Generating validation predictions for member 2...


C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Member 2 logits shape: (41833, 2)
Generating validation predictions for member 3...


C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Member 3 logits shape: (41833, 2)


In [9]:
val_member_probs = np.stack(
    val_member_probs,
    axis=0
)

print(
    "Validation member probability shape:",
    val_member_probs.shape
)

# Average probability across models
val_ensemble_probs = np.mean(
    val_member_probs,
    axis=0
)

print(
    "Validation ensemble probability shape:",
    val_ensemble_probs.shape
)

Validation member probability shape: (3, 41833)
Validation ensemble probability shape: (41833,)


In [10]:
def aggregate_predictions(
    probs,
    mapping,
    method="max",
    threshold=0.5
):

    mapping = (
        mapping.cpu().numpy()
        if torch.is_tensor(mapping)
        else np.asarray(mapping)
    )

    n_prompts = int(mapping.max()) + 1

    prompt_probs = np.zeros(n_prompts)

    for prompt_idx in range(n_prompts):

        chunk_probs = probs[
            mapping == prompt_idx
        ]

        if method == "max":

            prompt_probs[prompt_idx] = np.max(
                chunk_probs
            )

        elif method == "mean":

            prompt_probs[prompt_idx] = np.mean(
                chunk_probs
            )

        elif method == "majority":

            chunk_predictions = (
                chunk_probs >= threshold
            ).astype(int)

            prompt_probs[prompt_idx] = np.mean(
                chunk_predictions
            )

        else:

            raise ValueError(
                "method must be "
                "'max', 'mean', or 'majority'"
            )

    if method == "majority":

        prompt_predictions = (
            prompt_probs >= 0.5
        ).astype(int)

    else:

        prompt_predictions = (
            prompt_probs >= threshold
        ).astype(int)

    return prompt_probs, prompt_predictions

In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


def calculate_metrics(y_true, y_pred):

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0
    )

    fnr = (
        fn / (fn + tp)
        if (fn + tp) > 0
        else 0
    )

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": fpr,
        "FNR": fnr,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

In [12]:
validation_results = {}

for method in [
    "max",
    "mean",
    "majority"
]:

    _, val_predictions = aggregate_predictions(
        val_ensemble_probs,
        val_mapping,
        method=method
    )

    metrics = calculate_metrics(
        y_val,
        val_predictions
    )

    validation_results[method] = metrics


validation_table = pd.DataFrame(
    validation_results
).T


print("\n")
print("=" * 80)
print("BERT-TINY ENSEMBLE — PROMPT-LEVEL VALIDATION")
print("=" * 80)

print(
    validation_table[
        [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "FPR",
            "FNR"
        ]
    ].round(4)
)



BERT-TINY ENSEMBLE — PROMPT-LEVEL VALIDATION
          Accuracy  Precision  Recall      F1     FPR     FNR
max         0.9766     0.9742  0.9815  0.9779  0.0288  0.0185
mean        0.9776     0.9786  0.9788  0.9787  0.0238  0.0212
majority    0.9768     0.9750  0.9811  0.9780  0.0279  0.0189


In [13]:
best_method = validation_table[
    "F1"
].idxmax()

print("\nBest aggregation method:")
print(best_method)

print(
    "Validation F1:",
    validation_table.loc[
        best_method,
        "F1"
    ]
)


Best aggregation method:
mean
Validation F1: 0.9786642902075907


In [14]:
test_member_probs = []

for member_id, trainer in enumerate(
    trainers,
    start=1
):

    print(
        f"Generating test predictions "
        f"for member {member_id}..."
    )

    output = trainer.predict(
        test_dataset
    )

    logits = output.predictions

    probs = softmax(logits)[:, 1]

    test_member_probs.append(probs)

    print(
        f"Member {member_id} logits shape:",
        logits.shape
    )

Generating test predictions for member 1...


C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Member 1 logits shape: (41744, 2)
Generating test predictions for member 2...


C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Member 2 logits shape: (41744, 2)
Generating test predictions for member 3...


C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_22976\3149095571.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Member 3 logits shape: (41744, 2)


In [15]:
test_member_probs = np.stack(
    test_member_probs,
    axis=0
)

print(
    "Test member probability shape:",
    test_member_probs.shape
)

test_ensemble_probs = np.mean(
    test_member_probs,
    axis=0
)

print(
    "Test ensemble probability shape:",
    test_ensemble_probs.shape
)

Test member probability shape: (3, 41744)
Test ensemble probability shape: (41744,)


In [16]:
test_prompt_probs, test_predictions = (
    aggregate_predictions(
        test_ensemble_probs,
        test_mapping,
        method=best_method
    )
)

test_metrics = calculate_metrics(
    y_test,
    test_predictions
)

In [17]:
print("\n")
print("=" * 80)
print("BERT-TINY ENSEMBLE — FINAL TEST RESULTS")
print("=" * 80)

for metric, value in test_metrics.items():

    if metric in [
        "TN",
        "FP",
        "FN",
        "TP"
    ]:

        print(
            f"{metric:10s}: {value}"
        )

    else:

        print(
            f"{metric:10s}: {value:.4f}"
        )



BERT-TINY ENSEMBLE — FINAL TEST RESULTS
Accuracy  : 0.9765
Precision : 0.9776
Recall    : 0.9777
F1        : 0.9776
FPR       : 0.0248
FNR       : 0.0223
TN        : 16771
FP        : 427
FN        : 426
TP        : 18648


In [18]:
print("\nFINAL TEST METRICS (%)")
print("-" * 50)

for metric in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "FPR",
    "FNR"
]:

    print(
        f"{metric:10s}: "
        f"{test_metrics[metric] * 100:.2f}%"
    )


FINAL TEST METRICS (%)
--------------------------------------------------
Accuracy  : 97.65%
Precision : 97.76%
Recall    : 97.77%
F1        : 97.76%
FPR       : 2.48%
FNR       : 2.23%


In [19]:
print("\n")
print("=" * 80)
print("TEST SANITY CHECK")
print("=" * 80)

print(
    "Original test prompts :",
    len(test_df)
)

print(
    "Test chunks            :",
    len(test_mapping)
)

print(
    "Ensemble chunk probs   :",
    len(test_ensemble_probs)
)

print(
    "Prompt probabilities   :",
    len(test_prompt_probs)
)

print(
    "Prompt predictions     :",
    len(test_predictions)
)

print(
    "Ground-truth labels    :",
    len(y_test)
)



TEST SANITY CHECK
Original test prompts : 36272
Test chunks            : 41744
Ensemble chunk probs   : 41744
Prompt probabilities   : 36272
Prompt predictions     : 36272
Ground-truth labels    : 36272


In [21]:
ensemble_test_results = pd.DataFrame({

    "prompt": test_df["text"].to_numpy(),

    "true_label": y_test,

    "predicted_label": test_predictions,

    "malicious_probability": test_prompt_probs

})

ensemble_test_results.to_csv(
    "bert_tiny_ensemble_test_predictions.csv",
    index=False
)

print(
    "\nSaved:"
    " bert_tiny_ensemble_test_predictions.csv"
)


Saved: bert_tiny_ensemble_test_predictions.csv


In [22]:
validation_table.to_csv(
    "bert_tiny_ensemble_validation_results.csv"
)

print(
    "Saved:"
    " bert_tiny_ensemble_validation_results.csv"
)

Saved: bert_tiny_ensemble_validation_results.csv
